# GLiNER2 Classical and Retrieval Baseline Comparison

This notebook compares pure zero-shot GLiNER2 predictions with train-set retrieval and classical supervised TF-IDF baselines on Banking77.

Important: these methods are not all in the same setting. GLiNER2 `plain_label` and `mean_confidence_ensemble` are pure zero-shot if loaded from the schema wording notebook. TF-IDF kNN methods use the Banking77 train split as labeled retrieval memory. Logistic Regression and Linear SVM are supervised classical baselines trained on the Banking77 train split.


In [ ]:
PROJECT_NAME = "gliner2_classical_baselines_comparison"
DATASET_NAME = "mteb/banking77"
MODEL_NAME = "fastino/gliner2-base-v1"

MODE = "small"  # smoke / small / full
SEED = 42

SMOKE_N_EXAMPLES = 20
SMALL_PER_LABEL = 5
FULL_USE_ALL_TEST = True
CONFIRM_FULL_RUN = False

OUTPUT_DIR = "/content/gliner2_schema_outputs"
CLASSICAL_OUTPUT_SUBDIR = "classical_baselines_comparison"

FORCE_RERUN_CLASSICAL = False
LOAD_EXISTING_GLINER_RESULTS = True
RUN_GLINER_IF_MISSING = False

TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 1
TFIDF_MAX_FEATURES = 50000
TFIDF_SUBLINEAR_TF = True
TFIDF_LOWERCASE = True

K_NEIGHBORS = 15

LOGREG_C = 1.0
LOGREG_MAX_ITER = 3000

LINEAR_SVM_C = 1.0

RUN_BOOTSTRAP = True
BOOTSTRAP_SAMPLES = 1000

if MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "MODE='full' requires CONFIRM_FULL_RUN=True. This prevents accidental full evaluation."
    )


## Install/import dependencies


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR) / CLASSICAL_OUTPUT_SUBDIR
PREDICTIONS_DIR = OUTPUT_PATH / "predictions"
TABLES_DIR = OUTPUT_PATH / "tables"
FIGURES_DIR = OUTPUT_PATH / "figures"
for path in [OUTPUT_PATH, PREDICTIONS_DIR, TABLES_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()
    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    repo_url = "https://github.com/daidai-su/GLiNER2-demo.git"
    clone_dir = Path("/content/GLiNER2-demo")
    print("Project files were not found in the runtime.")
    print(f"Trying to clone project files from {repo_url} ...")
    try:
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
        PROJECT_ROOT = find_project_root()
    except Exception as exc:
        print(f"Automatic clone failed: {exc}")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Upload or clone the full repository into Colab."
    )

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
)


## Environment check


In [ ]:
import hashlib
import random
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from gliner2_project.baseline_comparison import (
    bootstrap_accuracy_ci,
    classical_results_summary,
    confusion_pairs,
    improved_degraded_vs_baseline,
    method_setting_table,
    oracle_analysis_frame,
    overlap_matrix,
    paired_comparison_frame,
    per_label_delta,
    per_label_metrics,
)
from gliner2_project.classical_baselines import (
    build_tfidf_vectorizer,
    run_tfidf_linear_svm,
    run_tfidf_logistic_regression,
)
from gliner2_project.data_utils import ensure_label_text_column, stratified_subset, unique_label_texts
from gliner2_project.env_utils import print_environment_info
from gliner2_project.knn_baselines import (
    assert_disjoint_example_ids,
    predict_tfidf_knn_majority,
    predict_tfidf_weighted_knn,
)
from gliner2_project.plotting import (
    plot_metric_bar,
    plot_overlap_matrix,
    plot_top_label_improvements,
)

random.seed(SEED)
np.random.seed(SEED)
RUN_START_UTC = datetime.now(timezone.utc).isoformat()
environment_info = print_environment_info()
print(f"Run started at: {RUN_START_UTC}")


## Load Banking77 train/test


In [ ]:
dataset = load_dataset(DATASET_NAME)
train_split = ensure_label_text_column(dataset["train"])
test_split = ensure_label_text_column(dataset["test"])


def split_to_examples(split, split_name, train_ids=False):
    rows = []
    for idx, row in enumerate(split):
        item = dict(row)
        item["example_id"] = f"{split_name}_{idx}" if train_ids else idx
        item["text"] = str(item["text"])
        item["label_text"] = str(item["label_text"])
        rows.append(item)
    return rows


train_examples = split_to_examples(train_split, "train", train_ids=True)
test_examples = split_to_examples(test_split, "test", train_ids=False)
label_texts = unique_label_texts(test_examples)
print(f"Train examples: {len(train_examples)}")
print(f"Test examples: {len(test_examples)}")
print(f"Labels: {len(label_texts)}")


## Build evaluation subset


In [ ]:
if MODE == "smoke":
    eval_examples = list(test_examples[:SMOKE_N_EXAMPLES])
elif MODE == "small":
    eval_examples = stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
elif MODE == "full":
    eval_examples = list(test_examples) if FULL_USE_ALL_TEST else stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
else:
    raise ValueError(f"Unknown MODE: {MODE}")

example_ids = [example["example_id"] for example in eval_examples]
assert_disjoint_example_ids([example["example_id"] for example in train_examples], example_ids)
print(f"Evaluation mode: {MODE}")
print(f"Evaluated examples: {len(eval_examples)}")
print(f"Methods to run directly: tfidf_knn_majority, tfidf_weighted_knn, tfidf_logistic_regression, tfidf_linear_svm")


## Save/load helpers


In [ ]:
def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=json_safe) + "\n")


def save_method_rows(method, rows, manifest=None):
    save_jsonl(rows, PREDICTIONS_DIR / f"{method}.jsonl")
    if manifest is not None:
        (PREDICTIONS_DIR / f"{method}.manifest.json").write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
            encoding="utf-8",
        )


def load_cached_classical_rows(method):
    path = PREDICTIONS_DIR / f"{method}.jsonl"
    if FORCE_RERUN_CLASSICAL or not path.exists():
        return None
    rows = load_jsonl(path)
    cached_ids = [str(row.get("example_id")) for row in rows]
    expected_ids = [str(example_id) for example_id in example_ids]
    if cached_ids != expected_ids:
        print(f"Ignoring cached {method}: evaluated example IDs differ.")
        return None
    print(f"Loaded cached {method}: {path}")
    return rows


## Load existing GLiNER2 result files if available


In [ ]:
predictions_by_method = {}
training_times = {}
total_prediction_times = {}


def align_cached_rows(rows, method):
    by_id = {str(row.get("example_id")): row for row in rows}
    aligned = []
    missing = []
    for example in eval_examples:
        key = str(example["example_id"])
        if key not in by_id:
            missing.append(example["example_id"])
            continue
        row = dict(by_id[key])
        row["example_id"] = example["example_id"]
        row["text"] = example["text"]
        row["gold_label"] = example["label_text"]
        row["method"] = method
        aligned.append(row)
    if missing:
        print(f"Cached {method} is unavailable for {len(missing)} evaluated examples.")
        return None
    return aligned


def try_load_existing_method(method, roots):
    for root in roots:
        path = Path(root) / "predictions" / f"{method}.jsonl"
        if not path.exists():
            continue
        rows = align_cached_rows(load_jsonl(path), method)
        if rows is not None:
            print(f"Loaded cached {method}: {path}")
            return rows
    return None


schema_roots = [Path(OUTPUT_DIR)]
retrieval_roots = [Path(OUTPUT_DIR) / "retrieval_pruning"]

if LOAD_EXISTING_GLINER_RESULTS:
    for method in ["plain_label", "mean_confidence_ensemble"]:
        rows = try_load_existing_method(method, schema_roots)
        if rows is not None:
            predictions_by_method[method] = rows
    for method in [
        "tfidf_candidate_pruning_gliner2",
        "tfidf_candidate_pruning_mean_confidence_ensemble",
    ]:
        rows = try_load_existing_method(method, retrieval_roots)
        if rows is not None:
            predictions_by_method[method] = rows

missing_gliner = [
    method
    for method in ["plain_label", "mean_confidence_ensemble"]
    if method not in predictions_by_method
]
print("Available cached GLiNER/retrieval-assisted methods:", sorted(predictions_by_method))
if missing_gliner and not RUN_GLINER_IF_MISSING:
    print("Missing GLiNER2 methods will be marked unavailable:", missing_gliner)


## Optional: recompute missing GLiNER2 methods


In [ ]:
if missing_gliner and RUN_GLINER_IF_MISSING:
    import torch

    from gliner2_project.ensemble import ENSEMBLE_INPUT_METHODS, mean_confidence_ensemble
    from gliner2_project.gliner2_wrappers import load_gliner2_model, predict_one_classification
    from gliner2_project.schema_variants import build_all_schema_variants

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = load_gliner2_model(MODEL_NAME, device=device)
    variants = build_all_schema_variants(label_texts, ENSEMBLE_INPUT_METHODS)
    schema_rows = {}
    for method, variant in variants.items():
        rows = []
        for example in tqdm(eval_examples, desc=f"GLiNER2 {method}"):
            rows.append(
                predict_one_classification(
                    model=model,
                    text=example["text"],
                    candidate_labels=list(variant["candidate_labels"]),
                    candidate_to_canonical=dict(variant["candidate_to_canonical"]),
                    method_name=method,
                    example_id=example["example_id"],
                    gold_label=example["label_text"],
                )
            )
        schema_rows[method] = rows
        if method == "plain_label":
            predictions_by_method["plain_label"] = rows
            save_method_rows("plain_label", rows)
    if "mean_confidence_ensemble" in missing_gliner:
        mean_rows = mean_confidence_ensemble(schema_rows)
        predictions_by_method["mean_confidence_ensemble"] = mean_rows
        save_method_rows("mean_confidence_ensemble", mean_rows)
elif missing_gliner:
    print("Skipping GLiNER2 recomputation because RUN_GLINER_IF_MISSING=False.")


## Build TF-IDF vectorizer on train split only


In [ ]:
train_texts = [example["text"] for example in train_examples]
train_labels = [example["label_text"] for example in train_examples]
train_ids = [example["example_id"] for example in train_examples]
eval_texts = [example["text"] for example in eval_examples]
gold_labels = [example["label_text"] for example in eval_examples]

vectorizer = build_tfidf_vectorizer(
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=TFIDF_SUBLINEAR_TF,
    lowercase=TFIDF_LOWERCASE,
)
train_tfidf = vectorizer.fit_transform(train_texts)
test_tfidf = vectorizer.transform(eval_texts)
print(f"TF-IDF train matrix: {train_tfidf.shape}")
print(f"TF-IDF eval matrix: {test_tfidf.shape}")


## Run TF-IDF kNN majority


In [ ]:
def attach_eval_metadata(method, payload_rows, total_prediction_time_sec, setting):
    average_latency = total_prediction_time_sec / len(payload_rows) if payload_rows else 0.0
    rows = []
    for example, payload in zip(eval_examples, payload_rows):
        row = dict(payload)
        row.update(
            {
                "example_id": example["example_id"],
                "text": example["text"],
                "gold_label": example["label_text"],
                "method": method,
                "setting": setting,
                "is_correct": row.get("predicted_canonical") == example["label_text"],
                "latency_sec": average_latency,
                "training_used": False,
                "train_texts_used": True,
                "train_labels_used": True,
            }
        )
        rows.append(row)
    return rows


method = "tfidf_knn_majority"
cached = load_cached_classical_rows(method)
if cached is None:
    start = time.perf_counter()
    payload = predict_tfidf_knn_majority(
        train_tfidf,
        train_labels,
        test_tfidf,
        k=K_NEIGHBORS,
        train_ids=train_ids,
    )
    total_time = time.perf_counter() - start
    rows = attach_eval_metadata(method, payload, total_time, "retrieval_only")
    save_method_rows(method, rows, {"total_prediction_time_sec": total_time, "k": K_NEIGHBORS})
else:
    rows = cached
    total_time = sum(float(row.get("latency_sec", 0.0)) for row in rows)
predictions_by_method[method] = rows
total_prediction_times[method] = total_time
print(f"{method}: {len(rows)} rows")


## Run TF-IDF similarity-weighted kNN


In [ ]:
method = "tfidf_weighted_knn"
cached = load_cached_classical_rows(method)
if cached is None:
    start = time.perf_counter()
    payload = predict_tfidf_weighted_knn(
        train_tfidf,
        train_labels,
        test_tfidf,
        k=K_NEIGHBORS,
        train_ids=train_ids,
    )
    total_time = time.perf_counter() - start
    rows = attach_eval_metadata(method, payload, total_time, "retrieval_only")
    save_method_rows(method, rows, {"total_prediction_time_sec": total_time, "k": K_NEIGHBORS})
else:
    rows = cached
    total_time = sum(float(row.get("latency_sec", 0.0)) for row in rows)
predictions_by_method[method] = rows
total_prediction_times[method] = total_time
print(f"{method}: {len(rows)} rows")


## Train and evaluate TF-IDF Logistic Regression


In [ ]:
def attach_supervised_metadata(result, examples, setting):
    rows = []
    for example, row in zip(examples, result.predictions):
        item = dict(row)
        item.update(
            {
                "example_id": example["example_id"],
                "text": example["text"],
                "setting": setting,
            }
        )
        rows.append(item)
    return rows


method = "tfidf_logistic_regression"
cached = load_cached_classical_rows(method)
if cached is None:
    result = run_tfidf_logistic_regression(
        train_texts=train_texts,
        train_labels=train_labels,
        test_texts=eval_texts,
        gold_labels=gold_labels,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        lowercase=TFIDF_LOWERCASE,
        C=LOGREG_C,
        max_iter=LOGREG_MAX_ITER,
        random_state=SEED,
    )
    rows = attach_supervised_metadata(result, eval_examples, "classical_supervised")
    training_times[method] = result.training_time_sec
    total_prediction_times[method] = result.total_prediction_time_sec
    save_method_rows(
        method,
        rows,
        {
            "training_time_sec": result.training_time_sec,
            "total_prediction_time_sec": result.total_prediction_time_sec,
            "config": result.config,
        },
    )
    print("Logistic Regression config:", result.config)
else:
    rows = cached
    total_prediction_times[method] = sum(float(row.get("latency_sec", 0.0)) for row in rows)
predictions_by_method[method] = rows
print(f"{method}: {len(rows)} rows")


## Train and evaluate TF-IDF Linear SVM


In [ ]:
method = "tfidf_linear_svm"
cached = load_cached_classical_rows(method)
if cached is None:
    result = run_tfidf_linear_svm(
        train_texts=train_texts,
        train_labels=train_labels,
        test_texts=eval_texts,
        gold_labels=gold_labels,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        lowercase=TFIDF_LOWERCASE,
        C=LINEAR_SVM_C,
        random_state=SEED,
    )
    rows = attach_supervised_metadata(result, eval_examples, "classical_supervised")
    training_times[method] = result.training_time_sec
    total_prediction_times[method] = result.total_prediction_time_sec
    save_method_rows(
        method,
        rows,
        {
            "training_time_sec": result.training_time_sec,
            "total_prediction_time_sec": result.total_prediction_time_sec,
            "config": result.config,
        },
    )
    print("Linear SVM config:", result.config)
else:
    rows = cached
    total_prediction_times[method] = sum(float(row.get("latency_sec", 0.0)) for row in rows)
predictions_by_method[method] = rows
print(f"{method}: {len(rows)} rows")


## Compare all available methods


In [ ]:
available_methods = list(predictions_by_method)
settings = method_setting_table(available_methods)
results_summary = classical_results_summary(
    predictions_by_method,
    training_times=training_times,
    total_prediction_times=total_prediction_times,
    method_settings=settings,
).sort_values("accuracy", ascending=False)

settings.to_csv(TABLES_DIR / "method_setting_table.csv", index=False)
results_summary.to_csv(TABLES_DIR / "classical_results_summary.csv", index=False)
display(results_summary)


## Paired comparison and error overlap analysis


In [ ]:
key_pairs = [
    ("plain_label", "tfidf_knn_majority"),
    ("plain_label", "tfidf_weighted_knn"),
    ("plain_label", "tfidf_logistic_regression"),
    ("plain_label", "tfidf_linear_svm"),
    ("mean_confidence_ensemble", "tfidf_knn_majority"),
    ("tfidf_knn_majority", "tfidf_weighted_knn"),
    ("tfidf_weighted_knn", "tfidf_logistic_regression"),
    ("tfidf_weighted_knn", "tfidf_linear_svm"),
]
available_pairs = [
    pair for pair in key_pairs if pair[0] in predictions_by_method and pair[1] in predictions_by_method
]
paired = paired_comparison_frame(predictions_by_method, available_pairs)
paired.to_csv(TABLES_DIR / "paired_comparisons.csv", index=False)
display(paired)

errors = overlap_matrix(predictions_by_method, correct=False)
corrects = overlap_matrix(predictions_by_method, correct=True)
errors.to_csv(TABLES_DIR / "error_overlap_matrix.csv")
corrects.to_csv(TABLES_DIR / "correct_overlap_matrix.csv")
display(errors)


## Bootstrap confidence intervals


In [ ]:
if RUN_BOOTSTRAP and available_pairs:
    bootstrap_samples = BOOTSTRAP_SAMPLES
    if MODE == "full" and BOOTSTRAP_SAMPLES > 1000:
        bootstrap_samples = 1000
    bootstrap_ci = bootstrap_accuracy_ci(
        predictions_by_method,
        available_pairs,
        n_samples=bootstrap_samples,
        seed=SEED,
        include_macro_f1=True,
    )
else:
    bootstrap_ci = pd.DataFrame()
bootstrap_ci.to_csv(TABLES_DIR / "bootstrap_accuracy_ci.csv", index=False)
display(bootstrap_ci)


## Oracle analysis


In [ ]:
oracle_groups = {
    "oracle_gliner_plain_plus_knn_majority": ["plain_label", "tfidf_knn_majority"],
    "oracle_gliner_plain_plus_weighted_knn": ["plain_label", "tfidf_weighted_knn"],
    "oracle_knn_majority_plus_weighted_knn": ["tfidf_knn_majority", "tfidf_weighted_knn"],
    "oracle_mean_confidence_plus_weighted_knn": ["mean_confidence_ensemble", "tfidf_weighted_knn"],
    "oracle_all_available_non_supervised": [
        "plain_label",
        "mean_confidence_ensemble",
        "tfidf_knn_majority",
        "tfidf_weighted_knn",
        "tfidf_candidate_pruning_gliner2",
        "tfidf_candidate_pruning_mean_confidence_ensemble",
    ],
}
oracle = oracle_analysis_frame(predictions_by_method, oracle_groups)
oracle.to_csv(TABLES_DIR / "oracle_analysis.csv", index=False)
display(oracle)


## Per-label analysis


In [ ]:
label_metrics = per_label_metrics(predictions_by_method)
label_metrics.to_csv(TABLES_DIR / "per_label_metrics.csv", index=False)

delta_frames = []
for baseline, comparison in [
    ("plain_label", "tfidf_weighted_knn"),
    ("plain_label", "tfidf_linear_svm"),
    ("tfidf_weighted_knn", "tfidf_linear_svm"),
]:
    if baseline in predictions_by_method and comparison in predictions_by_method:
        delta_frames.append(per_label_delta(label_metrics, baseline, comparison))

per_label_delta_vs_plain = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
per_label_delta_vs_plain.to_csv(TABLES_DIR / "per_label_delta_vs_plain.csv", index=False)
display(per_label_delta_vs_plain.head(30))


## Confusion analysis


In [ ]:
confusions = confusion_pairs(predictions_by_method, top_n=30)
confusions.to_csv(TABLES_DIR / "confusion_pairs.csv", index=False)
display(confusions.head(60))

comparison_for_examples = "tfidf_weighted_knn" if "tfidf_weighted_knn" in predictions_by_method else None
if comparison_for_examples and "plain_label" in predictions_by_method:
    improved, degraded = improved_degraded_vs_baseline(
        predictions_by_method,
        "plain_label",
        comparison_for_examples,
    )
else:
    improved, degraded = pd.DataFrame(), pd.DataFrame()
improved.to_csv(TABLES_DIR / "improved_examples_vs_plain.csv", index=False)
degraded.to_csv(TABLES_DIR / "degraded_examples_vs_plain.csv", index=False)
print(f"Improved vs plain_label: {len(improved)}")
print(f"Degraded vs plain_label: {len(degraded)}")


## Visualizations


In [ ]:
if not results_summary.empty:
    plot_metric_bar(
        results_summary,
        "accuracy",
        FIGURES_DIR / "classical_method_comparison_accuracy.png",
        "Method Accuracy Comparison",
    )
    plot_metric_bar(
        results_summary,
        "macro_f1",
        FIGURES_DIR / "classical_method_comparison_macro_f1.png",
        "Method Macro F1 Comparison",
    )

plot_overlap_matrix(
    errors,
    FIGURES_DIR / "error_overlap_matrix.png",
    "Error Overlap Matrix",
)

if not per_label_delta_vs_plain.empty:
    weighted_delta = per_label_delta_vs_plain[
        (per_label_delta_vs_plain["baseline_method"] == "plain_label")
        & (per_label_delta_vs_plain["comparison_method"] == "tfidf_weighted_knn")
    ]
    if not weighted_delta.empty:
        plot_top_label_improvements(
            weighted_delta,
            FIGURES_DIR / "top_label_improvements_weighted_knn_vs_plain.png",
            top_n=15,
        )
    svm_delta = per_label_delta_vs_plain[
        (per_label_delta_vs_plain["baseline_method"] == "plain_label")
        & (per_label_delta_vs_plain["comparison_method"] == "tfidf_linear_svm")
    ]
    if not svm_delta.empty:
        plot_top_label_improvements(
            svm_delta,
            FIGURES_DIR / "top_label_improvements_svm_vs_plain.png",
            top_n=15,
        )
print(f"Figures saved to: {FIGURES_DIR}")


## Save artifacts


In [ ]:
RUN_END_UTC = datetime.now(timezone.utc).isoformat()
manifest = {
    "project_name": PROJECT_NAME,
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "mode": MODE,
    "seed": SEED,
    "num_examples": len(eval_examples),
    "example_ids": example_ids,
    "start_time_utc": RUN_START_UTC,
    "end_time_utc": RUN_END_UTC,
    "tfidf_params": {
        "ngram_range": TFIDF_NGRAM_RANGE,
        "min_df": TFIDF_MIN_DF,
        "max_features": TFIDF_MAX_FEATURES,
        "sublinear_tf": TFIDF_SUBLINEAR_TF,
        "lowercase": TFIDF_LOWERCASE,
        "k_neighbors": K_NEIGHBORS,
    },
    "logistic_regression": {
        "C": LOGREG_C,
        "max_iter": LOGREG_MAX_ITER,
    },
    "linear_svm": {
        "C": LINEAR_SVM_C,
    },
    "bootstrap": {
        "run_bootstrap": RUN_BOOTSTRAP,
        "bootstrap_samples": BOOTSTRAP_SAMPLES,
    },
    "available_methods": available_methods,
    "environment": environment_info,
}
(OUTPUT_PATH / "classical_run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
    encoding="utf-8",
)
print(f"Artifacts saved under: {OUTPUT_PATH}")


## Final summary for report/presentation


In [ ]:
print("=== Classical baseline comparison summary ===")
display(results_summary)

print("\n=== Setting distinction ===")
display(settings)

print("\n=== Key paired comparisons ===")
display(paired)

print("\n=== Bootstrap CI ===")
display(bootstrap_ci)

print("\nRecommended framing:")
print("- GLiNER2 plain_label and mean_confidence_ensemble are pure zero-shot if loaded from schema wording outputs.")
print("- TF-IDF kNN baselines use Banking77 train split as labeled retrieval memory; do not call them zero-shot.")
print("- Logistic Regression and Linear SVM are supervised classical baselines trained on Banking77 train labels.")
print("- Use this notebook to show how strong simple train-set-based baselines are on Banking77.")
